# PathQ-Former — BLCA 5-Fold CV (Tuned Config)

**Changes from baseline:** Lower LR (5e-5), higher dropout (0.25), gradient accumulation (8), LR warmup (2 epochs), patience=3, stronger weight decay.

**Runtime:** `Runtime > Change runtime type > T4 GPU`

**Resume:** If disconnected, rerun all cells. Completed folds are skipped.

In [ ]:
# ============================================================
# Cell 1: Mount Drive
# ============================================================
from google.colab import drive
drive.mount('/content/drive')

import os
OUTPUT_DIR = '/content/drive/MyDrive/PathQFormer/outputs_tuned'
os.makedirs(OUTPUT_DIR, exist_ok=True)

progress_file = os.path.join(OUTPUT_DIR, 'blca', 'cv_progress.json')
if os.path.exists(progress_file):
    import json
    with open(progress_file) as f:
        prev = json.load(f)
    done = prev.get('fold_results', {})
    print(f'Previous progress: {len(done)}/5 folds complete')
    for fold, ci in sorted(done.items()):
        print(f'  Fold {fold}: C-Index = {ci:.4f}')
    if len(done) >= 5:
        print('\nAll folds done! Skip to Cell 7.')
else:
    print('Starting fresh')

In [ ]:
# ============================================================
# Cell 2: Check GPU and disk
# ============================================================
import torch, shutil

assert torch.cuda.is_available(), 'No GPU! Runtime > Change runtime type > T4'
print(f'GPU: {torch.cuda.get_device_name(0)}')
print(f'VRAM: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB')

total, used, free = shutil.disk_usage('/content')
print(f'Disk: {free / 1e9:.1f} GB free / {total / 1e9:.1f} GB total')

In [ ]:
# ============================================================
# Cell 3: Unzip source code + install deps
# ============================================================
os.chdir('/content')

# Always re-extract to get the latest code
ZIP_PATH = '/content/drive/MyDrive/PathQFormer/PathQFormer_src.zip'
if os.path.exists(ZIP_PATH):
    !unzip -qo "{ZIP_PATH}" -d /content/
else:
    raise FileNotFoundError(
        f'{ZIP_PATH} not found.\n'
        'Upload the updated PathQFormer_src.zip to Google Drive at MyDrive/PathQFormer/'
    )

os.chdir('/content/PathQFormer')
!pip install -q lifelines h5py pyyaml scikit-survival
print('Setup complete')

In [ ]:
# ============================================================
# Cell 4: Download BLCA embeddings
# ============================================================
import glob

EMBEDDINGS_DIR = '/content/embeddings_blca'
h5_files = glob.glob(f'{EMBEDDINGS_DIR}/**/*.h5', recursive=True)

if len(h5_files) < 400:
    !pip install -q huggingface_hub
    from huggingface_hub import login
    login()
    
    print('Downloading TCGA-BLCA embeddings (~30 GB)...')
    from huggingface_hub import hf_hub_download
    tar_path = hf_hub_download(
        repo_id='MahmoodLab/UNI2-h-features',
        filename='TCGA/TCGA-BLCA.tar.gz',
        repo_type='dataset',
        local_dir='/content/hf_cache',
    )
    
    print('Extracting...')
    os.makedirs(EMBEDDINGS_DIR, exist_ok=True)
    !tar -xzf "{tar_path}" -C "{EMBEDDINGS_DIR}"
    !rm -rf /content/hf_cache
    
    h5_files = glob.glob(f'{EMBEDDINGS_DIR}/**/*.h5', recursive=True)
    print(f'Done — {len(h5_files)} slides')
else:
    print(f'Embeddings present ({len(h5_files)} slides)')

if h5_files:
    EMBEDDINGS_DIR = os.path.dirname(h5_files[0])
print(f'Embeddings dir: {EMBEDDINGS_DIR}')

In [ ]:
# ============================================================
# Cell 5: Verify
# ============================================================
import h5py, pandas as pd

with h5py.File(h5_files[0], 'r') as f:
    print(f'Sample: {os.path.basename(h5_files[0])}')
    print(f'Shape: {f["features"].shape} | Dtype: {f["features"].dtype}')

meta = pd.read_csv('data/survpath_repo/datasets_csv/metadata/tcga_blca.csv')
print(f'BLCA metadata: {len(meta)} rows')

In [ ]:
# ============================================================
# Cell 6: Run 5-fold CV with TUNED config
# ============================================================
from src.training.train import train_cv

cfg = {
    # Data
    'survpath_dir': 'data/survpath_repo',
    'embeddings_dir': EMBEDDINGS_DIR,
    'output_dir': OUTPUT_DIR,
    'cancer_type': 'blca',
    'pathway_type': 'combine',
    
    # Architecture (unchanged)
    'wsi_input_dim': 1536,
    'hidden_dim': 256,
    'num_queries': 32,
    'num_heads': 8,
    'query_layers': 2,
    'fusion_layers': 2,
    'num_bins': 4,
    
    # Stronger regularization
    'dropout': 0.25,             # was 0.1
    'modality_dropout': 0.15,
    
    # Slower, more stable training
    'lr': 5e-5,                  # was 2e-4
    'weight_decay': 1e-4,        # was 1e-5
    'warmup_epochs': 2,          # NEW: linear warmup
    'grad_accum_steps': 8,       # NEW: effective batch size = 8
    
    'batch_size': 1,
    'epochs': 20,
    'patience': 3,               # was 5
    'num_folds': 5,
    'num_workers': 2,
    'nll_alpha': 0.0,
    
    # Pathway tokenizer
    'pathway_hidden_dim': 128,
    'min_genes': 3,
    'max_genes': 300,
    
    'max_patches': None,
}

results = train_cv(cfg)

In [ ]:
# ============================================================
# Cell 7: Results — compare baseline vs tuned
# ============================================================
import numpy as np, json

# Load tuned results
progress_file = os.path.join(OUTPUT_DIR, 'blca', 'cv_progress.json')
if os.path.exists(progress_file):
    with open(progress_file) as f:
        progress = json.load(f)
    results = [progress['fold_results'][str(i)] for i in range(5)]

# Baseline results
baseline = [0.5756, 0.6340, 0.5931, 0.6062, 0.6173]

print('='*60)
print('  PathQ-Former BLCA — Baseline vs Tuned')
print('='*60)
print(f'{"Fold":<8}{"Baseline":<12}{"Tuned":<12}{"Diff":<10}')
print('-'*42)
for i in range(5):
    diff = results[i] - baseline[i]
    print(f'{i:<8}{baseline[i]:<12.4f}{results[i]:<12.4f}{diff:+.4f}')
print('-'*42)
print(f'{"Mean":<8}{np.mean(baseline):<12.4f}{np.mean(results):<12.4f}{np.mean(results)-np.mean(baseline):+.4f}')
print(f'{"Std":<8}{np.std(baseline):<12.4f}{np.std(results):<12.4f}')
print('='*60)

# Save
results_path = os.path.join(OUTPUT_DIR, 'blca_results.json')
with open(results_path, 'w') as f:
    json.dump({
        'cancer_type': 'BLCA',
        'fold_cindices': results,
        'mean_cindex': float(np.mean(results)),
        'std_cindex': float(np.std(results)),
        'baseline_mean': float(np.mean(baseline)),
        'config': cfg,
    }, f, indent=2)
print(f'\nSaved to {results_path}')